In [1]:
import pandas as pd
import os
from os.path import dirname


root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [2]:
dataset = "bpic2015_5"

In [3]:
if(dataset == "BPI12_DECLINED_COMPLETE"):
    raw_data = pd.read_csv(f"datasets/original/{dataset}.csv")
else:
    raw_data = pd.read_csv(f"datasets/original/{dataset}.csv",sep=";")

raw_data.head()


/tmp/ipykernel_8480/1448010451.py:4: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(f"datasets/original/{dataset}.csv",sep=";")


,Responsible_actor,SUMleges,Case ID,Aanleg (Uitvoeren werk of werkzaamheid),Bouw,Brandveilig gebruik (melding),Brandveilig gebruik (vergunning),Flora en Fauna,Gebiedsbescherming,Handelen in strijd met regels RO,Inrit/Uitweg,Integraal,Kap,Milieu (melding),Milieu (neutraal wijziging),Milieu (omgevingsvergunning beperkte milieutoets),Milieu (vergunning),Monument,Reclame,Sloop,Activity,monitoringResource,question,Resource,Complete Timestamp,duration,month,weekday,hour,remtime,elapsed
0,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_180\\complete,560600,EMPTY,560429,2011-03-24 01:20:58,0.016667,3,3,1,0.0,14750458.0
1,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_150\\complete,560600,EMPTY,560429,2011-03-24 01:20:57,0.000000,3,3,1,1.0,14750457.0
2,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_110\\complete,560600,False,560429,2011-03-24 01:20:57,159380.950000,3,3,1,1.0,14750457.0
3,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_2\\complete,560600,EMPTY,560600,2010-12-03 09:00:00,528.283333,12,4,9,9562858.0,5187600.0
4,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_1\\complete,560600,EMPTY,560600,2010-12-03 00:11:43,100.866667,12,4,0,9594555.0,5155903.0


In [4]:
if dataset=="BPI12_DECLINED_COMPLETE":
    tab_all = raw_data.rename(
        columns={"case:concept:name": "CaseID", "concept:name": "Activity"}
    )
elif dataset == "BPIC15_1_f2":
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID", "label": "Label"}
    )
elif dataset in {"invoice", "sepsis", "Production_Data", "credit", "helpdesk"} or dataset.startswith("bpic2015"):
    tab_all = raw_data.rename(
         columns={"Complete Timestamp": "time:timestamp", "Case ID": "CaseID"}
    )
else:
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID", "concept:name": "Activity"}
    )



In [5]:
from datetime import datetime
import time

def translate_time(time_str):
    return datetime.fromisoformat(time_str).timestamp()

In [6]:
tab_all["time:timestamp"] = tab_all["time:timestamp"].apply(translate_time)

In [7]:
#To predict remaining cycle time we will add a new column.

#compute final timestamp for each trace.
tab_all["case_end_ts"] = (
    tab_all
    .groupby("CaseID")["time:timestamp"]
    .transform("max")
)

# add column remaining_time in seconds (float)
tab_all["remaining_time"] = (
    tab_all["case_end_ts"] - tab_all["time:timestamp"]
)

#clean up
tab_all.drop(columns=["case_end_ts"], inplace=True)

In [8]:
tab_all.head()

,Responsible_actor,SUMleges,CaseID,Aanleg (Uitvoeren werk of werkzaamheid),Bouw,Brandveilig gebruik (melding),Brandveilig gebruik (vergunning),Flora en Fauna,Gebiedsbescherming,Handelen in strijd met regels RO,Inrit/Uitweg,Integraal,Kap,Milieu (melding),Milieu (neutraal wijziging),Milieu (omgevingsvergunning beperkte milieutoets),Milieu (vergunning),Monument,Reclame,Sloop,Activity,monitoringResource,question,Resource,time:timestamp,duration,month,weekday,hour,remtime,elapsed,remaining_time
0,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_180\\complete,560600,EMPTY,560429,1.300915e+09,0.016667,3,3,1,0.0,14750458.0,0.0
1,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_150\\complete,560600,EMPTY,560429,1.300915e+09,0.000000,3,3,1,1.0,14750457.0,1.0
2,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_110\\complete,560600,False,560429,1.300915e+09,159380.950000,3,3,1,1.0,14750457.0,1.0
3,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_2\\complete,560600,EMPTY,560600,1.291352e+09,528.283333,12,4,9,9562858.0,5187600.0,9562858.0
4,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_1\\complete,560600,EMPTY,560600,1.291321e+09,100.866667,12,4,0,9594555.0,5155903.0,9594555.0


In [9]:
split_ratio = 8 / 10

first_act_tab = (
    tab_all.groupby("CaseID").first().sort_values("time:timestamp").reset_index()
)
first_act_tab = first_act_tab[
    ~first_act_tab.duplicated(subset=["CaseID", "Activity"], keep="first")
]
first_act_tab = first_act_tab.reset_index(drop=True)

list_train_valid_cases = list(
    first_act_tab[: int(split_ratio * len(first_act_tab))]["CaseID"].unique()
)

list_train_cases = list_train_valid_cases[: int(len(list_train_valid_cases) * 0.8)]
tab_train = tab_all[tab_all["CaseID"].isin(list_train_cases)].reset_index(drop=True)

list_valid_cases = list_train_valid_cases[int(len(list_train_valid_cases) * 0.8) :]
tab_valid = tab_all[tab_all["CaseID"].isin(list_valid_cases)].reset_index(drop=True)

list_test_cases = list(
    first_act_tab[int(split_ratio * len(first_act_tab)) :]["CaseID"].unique()
)
tab_test = tab_all[tab_all["CaseID"].isin(list_test_cases)].reset_index(drop=True)

In [10]:
tab_all.to_csv(data_dir_processed + f"{dataset}_processed_all.csv", index=False)

In [11]:
tab_train.to_csv(data_dir_processed+ f"{dataset}_processed_train.csv", index = False)

In [12]:
tab_valid.to_csv(data_dir_processed+f"{dataset}_processed_valid.csv", index = False)

In [13]:
tab_test.to_csv(data_dir_processed+ f"{dataset}_processed_test.csv", index = False)